In [115]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols


df = pd.read_csv("drom_cleaned.csv")

df['Город_крупный_регион'] = (
    df[['Город_Ленинградская область', 'Город_Московская область',
        'Город_Санкт-Петербург', 'Город_Москва', 'Город_Прочие города миллионники']]
    .max(axis=1)
)

df.drop(columns=[
    'Город_Ленинградская область', 'Город_Московская область',
    'Город_Санкт-Петербург', 'Город_Москва',
    'Город_Прочие города миллионники', 'Город_Другое'
], inplace=True)

df


,Стоимость,Модель,Город,Пробег,Объём двигателя,Тип топлива,Возраст мотоцикла,Коробка (код),Тактность (код),Подача топлива (код),...,Бренд_Honda,Бренд_Indian,Бренд_Kawasaki,Бренд_MV Agusta,Бренд_Royal Enfield,Бренд_Suzuki,Бренд_Triumph,Бренд_Yamaha,Бренд_Другое,Город_крупный_регион
0,12000.0,Иж 5,Курья,19000.0,350,бензин,27,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,12500.0,Kayo Basic YX125,Набережные Челны,500.0,125,бензин,0,1,1,0,...,0,0,0,0,0,0,0,0,1,0
2,15000.0,Иж Планета 4,Зерноград,999.0,350,бензин,26,0,1,0,...,0,0,0,0,0,0,0,0,1,0
3,15000.0,Урал 5557,Глазов,20000.0,620,бензин,38,0,1,0,...,0,0,0,0,0,0,0,0,1,0
4,15000.0,Восход 2М,Магнитогорск,10000.0,175,бензин,47,0,1,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8302,7650000.0,Harley-Davidson Tri Glide Ultra FLHTCUTG,Новосибирск,0.0,1868,бензин,1,0,1,1,...,0,0,0,0,0,0,0,0,0,1
8303,7950000.0,Harley-Davidson CVO Road Glide,Новосибирск,0.0,1983,бензин,1,0,1,1,...,0,0,0,0,0,0,0,0,0,1
8304,8250000.0,Harley-Davidson CVO Road Glide,Новосибирск,0.0,1977,бензин,0,0,1,1,...,0,0,0,0,0,0,0,0,0,1
8305,8500000.0,Harley-Davidson CVO Road Glide,Красноярск,0.0,1923,бензин,2,0,1,1,...,0,0,0,0,0,0,0,0,0,1


In [116]:
df['Бренд_топ'] = (
    df[['Бренд_Benelli', 'Бренд_CFMoto', 'Бренд_Ducati', 'Бренд_Harley-Davidson',
        'Бренд_Honda', 'Бренд_Indian', 'Бренд_Kawasaki', 'Бренд_MV Agusta',
        'Бренд_Royal Enfield', 'Бренд_Suzuki', 'Бренд_Triumph', 'Бренд_Yamaha']]
    .max(axis=1)
)

df.drop(columns=[
    'Бренд_Benelli', 'Бренд_CFMoto', 'Бренд_Ducati', 'Бренд_Harley-Davidson',
    'Бренд_Honda', 'Бренд_Indian', 'Бренд_Kawasaki', 'Бренд_MV Agusta',
    'Бренд_Royal Enfield', 'Бренд_Suzuki', 'Бренд_Triumph', 'Бренд_Yamaha',
    'Бренд_Другое', 'Модель', 'Город', 'Тип топлива'
], inplace=True)
df.columns = df.columns.str.replace(" ", "_").str.replace("-", "_")
df



,Стоимость,Пробег,Объём_двигателя,Возраст_мотоцикла,Коробка_(код),Тактность_(код),Подача_топлива_(код),Город_крупный_регион,Бренд_топ
0,12000.0,19000.0,350,27,0,0,0,0,0
1,12500.0,500.0,125,0,1,1,0,0,0
2,15000.0,999.0,350,26,0,1,0,0,0
3,15000.0,20000.0,620,38,0,1,0,0,0
4,15000.0,10000.0,175,47,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...
8302,7650000.0,0.0,1868,1,0,1,1,1,1
8303,7950000.0,0.0,1983,1,0,1,1,1,1
8304,8250000.0,0.0,1977,0,0,1,1,1,1
8305,8500000.0,0.0,1923,2,0,1,1,1,1


In [117]:
df = df.rename(columns={
    'Стоимость': 'Price',
    'Пробег': 'Mileage',
    'Объём_двигателя': 'EngineVolume',
    'Возраст_мотоцикла': 'Age',
    'Коробка_(код)': 'Gearbox',
    'Тактность_(код)': 'Stroke',
    'Подача_топлива_(код)': 'FuelSys'
})

df['log_Price'] = np.log(df['Price'])
df['log_Mileage'] = np.log1p(df['Mileage'])
df['log_EngineVolume'] = np.log1p(df['EngineVolume'])
df['log_Age'] = np.log1p(df['Age'])

brand_cols = [col for col in df.columns if col.startswith("Бренд_")]
region_cols = [col for col in df.columns if col.startswith("Город_")]

X1 = "Mileage + EngineVolume + Age"
X2 = X1 + " + Gearbox + Stroke + FuelSys"
X3 = X2 + " + " + " + ".join(brand_cols + region_cols)

X1_logX = "log_Mileage + log_EngineVolume + log_Age"
X2_logX = X1_logX + " + Gearbox + Stroke + FuelSys"
X3_logX = X2_logX + " + " + " + ".join(brand_cols + region_cols)

formulas = {
    "LINEAR_1": f"Price ~ {X1}",
    "LINEAR_2": f"Price ~ {X2}",
    "LINEAR_3": f"Price ~  {X3}",
    "SEMI_4": f"log_Price ~ {X1}",
    "SEMI_5": f"log_Price ~ {X2}",
    "SEMI_6": f"log_Price ~  {X3}",
    "LOG_7": f"log_Price ~ {X1_logX}",
    "LOG_8": f"log_Price ~ {X2_logX}",
    "LOG_9": f"log_Price ~ {X3_logX}",
}

results = []
equations = {}

for name, formula in formulas.items():
    model = ols(formula, data=df).fit()
    results.append({
        "Model": name,
        "R²": model.rsquared,
        "Adj. R²": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic
    })

    if name.startswith("SEMI"):
        lhs = "log(Y)"
    elif name.startswith("LOG"):
        lhs = "log(Y)"
    else:
        lhs = "Y"

    coeffs = model.params
    rhs = " + ".join([f"{v:.3f}·{k}" for k, v in coeffs.items()])
    equations[name] = f"{lhs} = {rhs}"

metrics_df = pd.DataFrame(results).round(3)

print("Сравнение моделей:")
print(metrics_df.to_string(index=False))

print("Уравнения моделей:")
for model_name in formulas:
    print(equations[model_name])

Сравнение моделей:
   Model    R²  Adj. R²        AIC        BIC
LINEAR_1 0.545    0.545 237412.686 237440.786
LINEAR_2 0.586    0.585 236634.468 236683.642
LINEAR_3 0.600    0.600 236340.471 236403.695
  SEMI_4 0.583    0.583  15311.701  15339.800
  SEMI_5 0.720    0.720  12010.191  12059.365
  SEMI_6 0.771    0.771  10323.351  10386.575
   LOG_7 0.660    0.660  13620.075  13648.174
   LOG_8 0.763    0.762  10636.817  10685.991
   LOG_9 0.807    0.807   8903.628   8966.851
Уравнения моделей:
Y = 96130.040·Intercept + -3.440·Mileage + 1057.156·EngineVolume + -11598.849·Age
Y = 270949.601·Intercept + -3.672·Mileage + 899.406·EngineVolume + -10173.685·Age + 113882.893·Gearbox + -225882.462·Stroke + 261109.787·FuelSys
Y = 298086.908·Intercept + -3.926·Mileage + 868.618·EngineVolume + -12939.049·Age + 100172.742·Gearbox + -277600.020·Stroke + 194242.701·FuelSys + 160274.662·Бренд_топ + 97304.995·Город_крупный_регион
log(Y) = 11.915·Intercept + 0.000·Mileage + 0.002·EngineVolume + -0.017·Ag

In [118]:
from scipy import stats
import patsy

print("\nТест Бокса-Кокса для модели LINEAR_3:")

formula = formulas["LINEAR_3"]

y, X = patsy.dmatrices(formula, data=df, return_type="dataframe")
y = y.squeeze()

valid = y > 0
y_valid = y[valid]
X_valid = X.loc[valid]

y_boxcox, lambda_bc = stats.boxcox(y_valid)

print(f"LINEAR_3: λ (lambda) = {lambda_bc:.4f}")




Тест Бокса-Кокса для модели LINEAR_3:
LINEAR_3: λ (lambda) = -0.0054


In [119]:
from statsmodels.stats.diagnostic import linear_reset

print("\nТест Рамсея (RESET) для модели LOG_9:")

formula = formulas["LOG_9"]
model = ols(formula, data=df).fit()

reset_result = linear_reset(model, power=2, use_f=True)

print(f"F-статистика: {reset_result.fvalue:.4f}")
print(f"p-value: {reset_result.pvalue:.4f}")

if reset_result.pvalue < 0.05:
    print("→ Отклоняем H0: модель специфицирована неверно.")
else:
    print("→ Не отклоняем H0: спецификация модели корректна.")



Тест Рамсея (RESET) для модели LOG_9:
F-статистика: 58.0099
p-value: 0.0000
→ Отклоняем H0: модель специфицирована неверно.


In [120]:
model_log9 = ols(formulas["LOG_9"], data=df).fit()

print(model_log9.summary())

lhs = "log(Y)"
coeffs = model_log9.params
rhs = " + ".join([f"{v:.3f}·{k}" for k, v in coeffs.items()])
equation = f"{lhs} = {rhs}"

print("\nУравнение модели LOG_9:")
print(equation)


                            OLS Regression Results                            
Dep. Variable:              log_Price   R-squared:                       0.807
Model:                            OLS   Adj. R-squared:                  0.807
Method:                 Least Squares   F-statistic:                     4346.
Date:                Thu, 08 May 2025   Prob (F-statistic):               0.00
Time:                        23:53:23   Log-Likelihood:                -4442.8
No. Observations:                8307   AIC:                             8904.
Df Residuals:                    8298   BIC:                             8967.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                8.7990 

In [121]:
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import shapiro, jarque_bera

print("\nПроверка предпосылок модели LOG_9:")

model_log9 = ols(formulas["LOG_9"], data=df).fit()
resid = model_log9.resid

_, pval_bp, _, _ = het_breuschpagan(resid, model_log9.model.exog)
print(f"Тест Бройша–Пагана: p-value = {pval_bp:.4f}")
if pval_bp < 0.05:
    print("→ Есть гетероскедастичность.")
else:
    print("→ Гомоскедастичность подтверждена.")

p_shapiro = shapiro(resid)[1]
p_jb = jarque_bera(resid)[1]
print(f"\nТест Шапиро–Уилка: p-value = {p_shapiro:.4f}")
print(f"Тест Харке–Бера: p-value = {p_jb:.4f}")
if p_shapiro > 0.05 and p_jb > 0.05:
    print("→ Остатки распределены нормально.")
else:
    print("→ Остатки НЕ распределены нормально.")

dw_stat = durbin_watson(resid)
print(f"\nКритерий Дарбина–Уотсона: DW = {dw_stat:.3f}")

print("\nVIF по признакам:")
exog = model_log9.model.exog
vif_vals = [variance_inflation_factor(exog, i) for i in range(exog.shape[1])]
for name, val in zip(model_log9.model.exog_names, vif_vals):
    print(f"{name}: {val:.2f}")



Проверка предпосылок модели LOG_9:
Тест Бройша–Пагана: p-value = 0.0000
→ Есть гетероскедастичность.

Тест Шапиро–Уилка: p-value = 0.0000
Тест Харке–Бера: p-value = 0.0000
→ Остатки НЕ распределены нормально.

Критерий Дарбина–Уотсона: DW = 1.284

VIF по признакам:
Intercept: 107.92
log_Mileage: 2.59
log_EngineVolume: 2.06
log_Age: 3.10
Gearbox: 1.01
Stroke: 1.13
FuelSys: 1.85
Бренд_топ: 2.10
Город_крупный_регион: 1.02


/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:586: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 8307.
  res = hypotest_fun_out(*samples, **kwds)


In [122]:
model_log9_robust = ols(formulas["LOG_9"], data=df).fit(cov_type='HC3')

print("\nLOG_9 с робастными стандартными ошибками (HC3):")
print(model_log9_robust.summary())

lhs = "log(Y)"
coeffs = model_log9_robust.params
rhs = " + ".join([f"{v:.3f}·{k}" for k, v in coeffs.items()])
equation_robust = f"{lhs} = {rhs}"

print("\nУравнение модели LOG_9 (с учетом робастных ошибок):")
print(equation_robust)




LOG_9 с робастными стандартными ошибками (HC3):
                            OLS Regression Results                            
Dep. Variable:              log_Price   R-squared:                       0.807
Model:                            OLS   Adj. R-squared:                  0.807
Method:                 Least Squares   F-statistic:                     5209.
Date:                Thu, 08 May 2025   Prob (F-statistic):               0.00
Time:                        23:53:23   Log-Likelihood:                -4442.8
No. Observations:                8307   AIC:                             8904.
Df Residuals:                    8298   BIC:                             8967.
Df Model:                           8                                         
Covariance Type:                  HC3                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------

In [123]:
# Удаляем переменную Gearbox из формулы LOG_9
formula_reduced = formulas["LOG_9"].replace("+ Gearbox", "").replace(" +Gearbox", "")

# Обучаем новую модель без Gearbox
model_log9_reduced = ols(formula_reduced, data=df).fit(cov_type='HC3')

print("\nМодель LOG_9 без переменной Gearbox (с робастными ошибками HC3):")
print(model_log9_reduced.summary())

# Формируем уравнение
lhs = "log(Y)"
coeffs = model_log9_reduced.params
rhs = " + ".join([f"{v:.3f}·{k}" for k, v in coeffs.items()])
equation_reduced = f"{lhs} = {rhs}"

print("\nУравнение модели без Gearbox:")
print(equation_reduced)



Модель LOG_9 без переменной Gearbox (с робастными ошибками HC3):
                            OLS Regression Results                            
Dep. Variable:              log_Price   R-squared:                       0.807
Model:                            OLS   Adj. R-squared:                  0.807
Method:                 Least Squares   F-statistic:                     5941.
Date:                Thu, 08 May 2025   Prob (F-statistic):               0.00
Time:                        23:53:23   Log-Likelihood:                -4443.0
No. Observations:                8307   AIC:                             8902.
Df Residuals:                    8299   BIC:                             8958.
Df Model:                           7                                         
Covariance Type:                  HC3                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------

In [124]:
from statsmodels.stats.diagnostic import linear_reset

print("\nПОИСК ФУНКЦИОНАЛЬНОЙ ФОРМЫ ДЛЯ ПРОХОЖДЕНИЯ ТЕСТА РАМСЕЯ\n")

variants = {
    "LOG_9": formulas["LOG_9"],

    # 1. Добавим квадратичные термы
    "V1_quad": formulas["LOG_9"] + " + np.power(log_Mileage, 2) + np.power(log_EngineVolume, 2) + np.power(log_Age, 2)",

    # 2. Взаимодействия между непрерывными переменными
    "V2_inter": formulas["LOG_9"] + " + log_Mileage:log_EngineVolume + log_EngineVolume:log_Age",

    # 3. И логарифм + квадраты + взаимодействие (расширенная спецификация)
    "V3_full": formulas["LOG_9"] + " + np.power(log_Mileage, 2) + log_Mileage:log_EngineVolume + np.power(log_EngineVolume, 2) + log_EngineVolume:log_Age",

    # 4. Преобразование категориальных переменных в взаимодействия
    "V4_inter_cat": formulas["LOG_9"] + " + FuelSys:Stroke + FuelSys:Бренд_топ + Город_крупный_регион:log_Age",

    # 5. Удалим переменные с высоким p-value (Gearbox, Stroke), упростим
    "V5_simplified": "log_Price ~ log_Mileage + log_EngineVolume + log_Age + FuelSys + Бренд_топ + Город_крупный_регион"
}

for name, formula in variants.items():
    try:
        model = ols(formula, data=df).fit()
        reset = linear_reset(model, power=2, use_f=True)
        pval = reset.pvalue
        print(f"{name}: RESET p-value = {pval:.4f}")
        if pval < 0.05:
            print("→ Модель НЕ проходит тест Рамсея.\n")
        else:
            print("→ Модель проходит тест Рамсея \n")
    except Exception as e:
        print(f"{name}: Ошибка — {e}\n")



ПОИСК ФУНКЦИОНАЛЬНОЙ ФОРМЫ ДЛЯ ПРОХОЖДЕНИЯ ТЕСТА РАМСЕЯ

LOG_9: RESET p-value = 0.0000
→ Модель НЕ проходит тест Рамсея.

V1_quad: RESET p-value = 0.0000
→ Модель НЕ проходит тест Рамсея.

V2_inter: RESET p-value = 0.0000
→ Модель НЕ проходит тест Рамсея.

V3_full: RESET p-value = 0.0000
→ Модель НЕ проходит тест Рамсея.

V4_inter_cat: RESET p-value = 0.0000
→ Модель НЕ проходит тест Рамсея.

V5_simplified: RESET p-value = 0.0000
→ Модель НЕ проходит тест Рамсея.



In [125]:
variants.update({
    # 6. Кубические термы
    "V6_cubic": formulas["LOG_9"] + " + np.power(log_Mileage, 3) + np.power(log_EngineVolume, 3) + np.power(log_Age, 3)",

    # 7. Инверсные переменные
    "V7_inverse": formulas["LOG_9"] + " + 1/log_Mileage + 1/log_EngineVolume + 1/log_Age",

    # 8. Логарифмы отношений
    "V8_ratio_logs": formulas["LOG_9"] + " + np.log(log_Mileage / log_Age + 0.01) + np.log(log_EngineVolume / log_Age + 0.01)",

    # 9. Dummy-переменная: старше 10 лет
    "V9_step_age": formulas["LOG_9"] + " + (Age > 10).astype(int)",

    # 10. Возраст как категории (биннинг)
    "V10_binned_age": formulas["LOG_9"] + " + pd.cut(Age, bins=[0,5,10,15,30,50], labels=False)"
})

for name in list(variants)[-5:]:
    try:
        model = ols(variants[name], data=df).fit()
        reset = linear_reset(model, power=2, use_f=True)
        pval = reset.pvalue
        print(f"{name}: RESET p-value = {pval:.4f}")
        if pval < 0.05:
            print("→ Модель НЕ проходит тест Рамсея.\n")
        else:
            print("→ Модель проходит тест Рамсея \n")
    except Exception as e:
        print(f"{name}: Ошибка — {e}\n")


V6_cubic: RESET p-value = 0.0000
→ Модель НЕ проходит тест Рамсея.

V7_inverse: Ошибка — intercept term cannot interact with anything else
    log_Price ~ log_Mileage + log_EngineVolume + log_Age + Gearbox + Stroke + FuelSys + Бренд_топ + Город_крупный_регион + 1/log_Mileage + 1/log_EngineVolume + 1/log_Age
                                                                                                                           ^

V8_ratio_logs: Ошибка — exog contains inf or nans

V9_step_age: Ошибка — expected an operator, not '.astype(int)'
    log_Price ~ log_Mileage + log_EngineVolume + log_Age + Gearbox + Stroke + FuelSys + Бренд_топ + Город_крупный_регион + (Age > 10).astype(int)
                                                                                                                                     ^^^^^^^^^^^^

V10_binned_age: RESET p-value = 0.0000
→ Модель НЕ проходит тест Рамсея.



In [126]:
new_bike = pd.DataFrame([{
    "Age": 5,
    "Mileage": 11000,
    "EngineVolume": 700,
    "Stroke": 1,                # 4-тактный (код 1)
    "FuelSys": 1,               # инжектор
    "Бренд_топ": 1,             # значимый бренд
    "Город_крупный_регион": 1  # Москва — крупный регион
}])

new_bike["log_Mileage"] = np.log1p(new_bike["Mileage"])
new_bike["log_EngineVolume"] = np.log1p(new_bike["EngineVolume"])
new_bike["log_Age"] = np.log1p(new_bike["Age"])

predictors = model_log9_reduced.model.exog_names
if "Intercept" in predictors:
    predictors.remove("Intercept")

X_new = sm.add_constant(new_bike[predictors], has_constant='add')

log_price_pred = model_log9_reduced.predict(X_new)[0]
price_pred = np.exp(log_price_pred)

print(f"Логарифм прогнозной цены: {log_price_pred:.3f}")
print(f"Прогнозная цена: {price_pred:,.0f} руб.")


Логарифм прогнозной цены: 13.739
Прогнозная цена: 926,199 руб.
